# 💊 Классификация таблеток по изображениям

## О проекте

В этом проекте решается задача многоклассовой классификации изображений таблеток с помощью методов компьютерного зрения и transfer learning.

Модель получает фотографию таблетки и предсказывает её класс. В качестве основы используется предобученная `MobileNetV3-small`, после чего выполняется fine-tuning части сети.

**Итоговый результат:** accuracy **77.78%** на валидационной выборке.

> Датасет не включён в публичный репозиторий. Для запуска необходимо разместить данные локально в директории `dataset/`.


## Подготовка данных

Датасет не публикуется в репозитории. Перед запуском ноутбука необходимо разместить изображения локально в директории `dataset/`, сохранив структуру обучающей и валидационной выборок.


In [ ]:
from pathlib import Path

DATASET_DIR = Path("/content/dataset")

if not DATASET_DIR.exists():
    print("Директория dataset не найдена. Добавьте датасет перед запуском обучения.")
else:
    print("Датасет найден:", DATASET_DIR)


In [ ]:
# Ожидаемая структура данных:
# dataset/
# ├── train/
# │   ├── class_1/
# │   ├── class_2/
# │   └── ...
# └── val/
#     ├── class_1/
#     ├── class_2/
#     └── ...


In [ ]:
if DATASET_DIR.exists():
    for path in sorted(DATASET_DIR.iterdir()):
        if path.is_dir():
            print(path)


In [ ]:
# Этап 1. Загрузка и предобработка данных

import torch
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms
from matplotlib import pyplot as plt

# Трансформации для обучающего датасета
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.2),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=(-5, 5), fill=255),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Трансформации для проверочного датасета
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Загрузка датасетов
train_dataset = ImageFolder(
    root='/content/dataset/ogyeiv2/train',
    transform=train_transforms
)

val_dataset = ImageFolder(
    root='/content/dataset/ogyeiv2/test',
    transform=val_transforms
)

# Упаковка датасетов в DataLoader
batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)


In [ ]:
# Результаты этапа 1

classes = train_dataset.classes
num_classes = len(classes)

print("Количество классов:", num_classes)
print("Количество изображений в train:", len(train_dataset))
print("Количество изображений в val:", len(val_dataset))
print("Список классов:", classes)

print("\nПеременная train_loader:", train_loader)
print("Переменная val_loader:", val_loader)


Количество классов: 84
Количество изображений в train: 2352
Количество изображений в val: 504
Список классов: ['acc_long_600_mg', 'advil_ultra_forte', 'akineton_2_mg', 'algoflex_forte_dolo_400_mg', 'algoflex_rapid_400_mg', 'algopyrin_500_mg', 'ambroxol_egis_30_mg', 'apranax_550_mg', 'aspirin_ultra_500_mg', 'atoris_20_mg', 'atorvastatin_teva_20_mg', 'betaloc_50_mg', 'bila_git', 'c_vitamin_teva_500_mg', 'calci_kid', 'cataflam_50_mg', 'cataflam_dolo_25_mg', 'cetirizin_10_mg', 'cold_fx', 'coldrex', 'concor_10_mg', 'concor_5_mg', 'condrosulf_800_mg', 'controloc_20_mg', 'covercard_plus_10_mg_2_5_mg_5_mg', 'coverex_4_mg', 'diclopram_75-mg_20-mg', 'dorithricin_mentol', 'dulsevia_60_mg', 'enterol_250_mg', 'favipiravir_meditop_200_mg', 'ibumax_400_mg', 'jutavit_c_vitamin', 'jutavit_cink', 'kalcium_magnezium_cink', 'kalium_r', 'koleszterin_kontroll', 'lactamed', 'lactiv_plus', 'laresin_10_mg', 'letrox_50_mikrogramm', 'lordestin_5_mg', 'merckformin_xr_1000_mg', 'meridian', 'metothyrin_10_mg', 'mez

### Вывод по этапу 1

На этом этапе изображения приводятся к размеру 224×224 пикселя и нормализуются. Для обучающего датасета дополнительно применяются аугментации, использованные в уроках: случайные горизонтальные и вертикальные отражения и небольшой поворот. Проверочный датасет не аугментируется.

В результате подготовлены `train_loader` и `val_loader`, которые будут использоваться при обучении и оценке модели.


In [ ]:
# Этап 2. Объявление модели

import torch.nn as nn
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
from torchsummary import summary

# Загружаем MobileNetV3-small с предобученными весами ImageNet
model = mobilenet_v3_small(
    weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1
)

# Заменяем исходный классификатор на полносвязный слой
# с количеством выходов, равным количеству классов таблеток
in_features = model.classifier[0].in_features

model.classifier = nn.Linear(
    in_features=in_features,
    out_features=num_classes
)

# Замораживаем все веса модели
for param in model.parameters():
    param.requires_grad = False

# Размораживаем новый классификатор
for param in model.classifier.parameters():
    param.requires_grad = True


Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 104MB/s]


In [ ]:
# Результаты этапа 2

print(model)

print("\nКоличество классов на выходе:", model.classifier.out_features)

trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)
frozen_params = sum(
    p.numel() for p in model.parameters() if not p.requires_grad
)

print("Обучаемых параметров:", trainable_params)
print("Замороженных параметров:", frozen_params)


MobileNetV3(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): Hardswish()
    )
    (1): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=16, bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (1): SqueezeExcitation(
          (avgpool): AdaptiveAvgPool2d(output_size=1)
          (fc1): Conv2d(16, 8, kernel_size=(1, 1), stride=(1, 1))
          (fc2): Conv2d(8, 16, kernel_size=(1, 1), stride=(1, 1))
          (activation): ReLU()
          (scale_activation): Hardsigmoid()
        )
        (2): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(1, 1), 

### Вывод по этапу 2

Для классификации используется предобученная на ImageNet модель `MobileNetV3-small`. Все веса предобученной части модели заморожены. Исходный блок классификации заменён на один полносвязный слой, число выходов которого соответствует количеству классов таблеток. Обучаться будет только новый классификатор.


In [ ]:
# Этап 3. Обучение или дообучение

import torch.optim as optim

# Выбор устройства
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Устройство:", device)

model.to(device)

# Функция потерь и оптимизатор
loss_fn = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.001
)

EPOCHS = 10

# Код обучения для одной эпохи
def train_one_epoch(epoch_index):
    model.train()

    running_loss = 0.
    last_loss = 0.

    for batch_index, data in enumerate(train_loader):
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)

        loss = loss_fn(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        # Средняя ошибка за последние 20 батчей
        if (batch_index + 1) % 20 == 0:
            last_loss = running_loss / 20
            running_loss = 0.

    # Если в эпохе оказалось меньше 20 батчей
    if last_loss == 0.:
        last_loss = running_loss / max(1, len(train_loader))

    return last_loss


Устройство: cuda


In [ ]:
# Цикл обучения и сохранение лучшей модели

best_vloss = float('inf')

for epoch in range(EPOCHS):
    print(f'Эпоха {epoch + 1}')

    # Обучение
    avg_loss = train_one_epoch(epoch)

    # Валидация
    model.eval()

    running_vloss = 0.

    with torch.no_grad():
        for data in val_loader:
            inputs, labels = data
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)

            vloss = loss_fn(outputs, labels)
            running_vloss += vloss.item()

    avg_vloss = running_vloss / len(val_loader)

    # Сохраняем модель с минимальной ошибкой на валидации
    if avg_vloss < best_vloss:
        best_vloss = avg_vloss
        torch.save(
            model.state_dict(),
            'meds_classifier.pt'
        )

    print(
        f'В конце эпохи ошибка train {avg_loss:.4f}, '
        f'ошибка val {avg_vloss:.4f}'
    )


Эпоха 1
В конце эпохи ошибка train 4.1034, ошибка val 4.3295
Эпоха 2
В конце эпохи ошибка train 3.5183, ошибка val 4.3373
Эпоха 3
В конце эпохи ошибка train 3.0575, ошибка val 4.3980
Эпоха 4
В конце эпохи ошибка train 2.7854, ошибка val 4.3044
Эпоха 5
В конце эпохи ошибка train 2.4466, ошибка val 3.6332
Эпоха 6
В конце эпохи ошибка train 2.2510, ошибка val 2.7904
Эпоха 7
В конце эпохи ошибка train 2.0796, ошибка val 2.1527
Эпоха 8
В конце эпохи ошибка train 1.9330, ошибка val 1.8432
Эпоха 9
В конце эпохи ошибка train 1.8075, ошибка val 1.6481
Эпоха 10
В конце эпохи ошибка train 1.7321, ошибка val 1.5379


In [ ]:
# Результаты этапа 3

print("Минимальная ошибка на валидации:", best_vloss)
print("Модель сохранена в файл meds_classifier.pt")


Минимальная ошибка на валидации: 1.5378752015531063
Модель сохранена в файл meds_classifier.pt


### Вывод по этапу 3

Модель дообучается на GPU, если CUDA доступна, иначе используется CPU. В конце каждой эпохи выводятся ошибки на обучающем и проверочном датасетах. При улучшении ошибки на проверочном датасете веса модели сохраняются в файл `meds_classifier.pt`.


In [ ]:
# Fine-tuning модели

# Возвращаем модель на GPU
model.to(device)

# Размораживаем последний блок признаков MobileNetV3
for param in model.features[-1].parameters():
    param.requires_grad = True

# Новый оптимизатор с меньшим learning rate
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.0001
)

FINE_TUNE_EPOCHS = 5

best_vloss = float('inf')

for epoch in range(FINE_TUNE_EPOCHS):
    print(f'Fine-tuning, эпоха {epoch + 1}')

    # Обучение
    avg_loss = train_one_epoch(epoch)

    # Валидация
    model.eval()

    running_vloss = 0.

    with torch.no_grad():
        for data in val_loader:
            inputs, labels = data

            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)

            vloss = loss_fn(outputs, labels)
            running_vloss += vloss.item()

    avg_vloss = running_vloss / len(val_loader)

    # Сохраняем лучшую модель
    if avg_vloss < best_vloss:
        best_vloss = avg_vloss

        torch.save(
            model.state_dict(),
            'meds_classifier.pt'
        )

        print('Сохранена новая лучшая модель')

    print(
        f'В конце эпохи ошибка train {avg_loss:.4f}, '
        f'ошибка val {avg_vloss:.4f}'
    )

Fine-tuning, эпоха 1
Сохранена новая лучшая модель
В конце эпохи ошибка train 1.6004, ошибка val 1.4455
Fine-tuning, эпоха 2
Сохранена новая лучшая модель
В конце эпохи ошибка train 1.4890, ошибка val 1.3864
Fine-tuning, эпоха 3
Сохранена новая лучшая модель
В конце эпохи ошибка train 1.4323, ошибка val 1.3338
Fine-tuning, эпоха 4
Сохранена новая лучшая модель
В конце эпохи ошибка train 1.4694, ошибка val 1.2887
Fine-tuning, эпоха 5
Сохранена новая лучшая модель
В конце эпохи ошибка train 1.3517, ошибка val 1.2456


In [ ]:
# Этап 4. Оценка качества

from sklearn.metrics import classification_report, accuracy_score
import numpy as np

# Загружаем лучшую сохранённую модель
model.load_state_dict(
    torch.load(
        'meds_classifier.pt',
        map_location='cpu'
    )
)

model.to('cpu')
model.eval()

labels_predicted = []
labels_true = []

with torch.no_grad():
    for data in val_loader:
        inputs, labels = data

        outputs = model(inputs)

        predicted = torch.argmax(
            outputs,
            dim=1
        )

        preds = predicted.cpu().numpy()

        labels_predicted.extend(preds)
        labels_true.extend(labels.cpu().numpy())

labels_predicted = np.array(labels_predicted)
labels_true = np.array(labels_true)


In [ ]:
# Метрики качества

print(
    classification_report(
        labels_true,
        labels_predicted,
        target_names=classes,
        digits=4
    )
)

overall_accuracy = accuracy_score(
    labels_true,
    labels_predicted
)

print(f"Общая accuracy: {overall_accuracy:.4f}")


                                  precision    recall  f1-score   support

                 acc_long_600_mg     1.0000    1.0000    1.0000         6
               advil_ultra_forte     1.0000    1.0000    1.0000         6
                   akineton_2_mg     1.0000    0.8333    0.9091         6
      algoflex_forte_dolo_400_mg     1.0000    0.8333    0.9091         6
           algoflex_rapid_400_mg     1.0000    1.0000    1.0000         6
                algopyrin_500_mg     0.7143    0.8333    0.7692         6
             ambroxol_egis_30_mg     1.0000    1.0000    1.0000         6
                  apranax_550_mg     0.6000    1.0000    0.7500         6
            aspirin_ultra_500_mg     1.0000    0.5000    0.6667         6
                    atoris_20_mg     1.0000    0.8333    0.9091         6
         atorvastatin_teva_20_mg     0.6250    0.8333    0.7143         6
                   betaloc_50_mg     0.8571    1.0000    0.9231         6
                        bila_git     

In [ ]:
# Анализ ошибок по классам

errors_by_class = {}

for class_index, class_name in enumerate(classes):
    class_mask = labels_true == class_index

    errors = (
        labels_predicted[class_mask] != labels_true[class_mask]
    ).sum()

    errors_by_class[class_name] = int(errors)

sorted_errors = sorted(
    errors_by_class.items(),
    key=lambda x: x[1],
    reverse=True
)

worst_5_classes = sorted_errors[:5]

classes_without_errors = [
    class_name
    for class_name, errors in sorted_errors
    if errors == 0
]

print("5 классов с наибольшим количеством ошибок:")
for class_name, errors in worst_5_classes:
    print(f"{class_name}: {errors}")

print("\nКлассы без ошибок:")
if classes_without_errors:
    for class_name in classes_without_errors:
        print(class_name)
else:
    print("Таких классов нет.")


5 классов с наибольшим количеством ошибок:
teva_ambrobene_30_mg: 6
covercard_plus_10_mg_2_5_mg_5_mg: 5
nebivolol_sandoz_5_mg: 5
quamatel_40_mg: 4
rubophen_500_mg: 4

Классы без ошибок:
acc_long_600_mg
advil_ultra_forte
algoflex_rapid_400_mg
ambroxol_egis_30_mg
apranax_550_mg
betaloc_50_mg
bila_git
c_vitamin_teva_500_mg
calci_kid
cataflam_dolo_25_mg
cetirizin_10_mg
concor_10_mg
coverex_4_mg
dorithricin_mentol
dulsevia_60_mg
jutavit_c_vitamin
kalium_r
lactamed
letrox_50_mikrogramm
naprosyn_250_mg
normodipine_5_mg
ocutein
salazopyrin_en_500_mg
semicillin_500_mg
sinupret_forte
strepfen_8_75_mg
tricovel_tricoage45
urzinol
valeriana_teva
verospiron_25_mg
vitamin_d3_fresenius_1000_ne
voltaren_dolo_rapid_25_mg


In [ ]:
# Анализ результатов и ошибок модели

worst_names = [name for name, _ in worst_5_classes]

print("1. На каких 5 классах модель ошибается чаще всего?")
print(", ".join(worst_names))

print("\n2. Почему модель может ошибаться на этих классах?")
print(
    "Ошибки могут быть связаны с визуальным сходством таблеток разных классов, "
    "а также с различиями в освещении и ракурсе съёмки изображений."
)

print("\n3. На каких классах модель не совершает ошибок?")
if classes_without_errors:
    print(", ".join(classes_without_errors))
else:
    print("На проверочном датасете нет классов, распознанных полностью без ошибок.")

print("\n4. Почему эти классы модель распознаёт безошибочно?")
print(
    "Такие классы могут иметь хорошо различимые визуальные признаки, "
    "которые модель устойчиво выделяет на изображениях."
)

print("\n5. Как можно улучшить точность классификатора?")
print(
    "Точность можно дополнительно повысить, продолжив дообучение модели, увеличив объём обучающих данных или подобрав параметры аугментации,"
    "Также можно разморозить больше последних слоёв предобученной сети и выполнить более глубокий fine-tuning.."
)

print("\n6. Как ещё можно проанализировать результаты и ошибки модели?")
print(
    "Можно изучить матрицу ошибок и отдельно посмотреть изображения, "
    "на которых модель сделала неверные предсказания."
)

print(f"\nИтоговая accuracy классификатора: {overall_accuracy:.4f}")

if overall_accuracy > 0.75:
    print("Требование проекта выполнено: accuracy выше 75%.")
else:
    print("Требование проекта пока не выполнено: необходимо повысить accuracy выше 75%.")


1. На каких 5 классах модель ошибается чаще всего?
teva_ambrobene_30_mg, covercard_plus_10_mg_2_5_mg_5_mg, nebivolol_sandoz_5_mg, quamatel_40_mg, rubophen_500_mg

2. Почему модель может ошибаться на этих классах?
Ошибки могут быть связаны с визуальным сходством таблеток разных классов, а также с различиями в освещении и ракурсе съёмки изображений.

3. На каких классах модель не совершает ошибок?
acc_long_600_mg, advil_ultra_forte, algoflex_rapid_400_mg, ambroxol_egis_30_mg, apranax_550_mg, betaloc_50_mg, bila_git, c_vitamin_teva_500_mg, calci_kid, cataflam_dolo_25_mg, cetirizin_10_mg, concor_10_mg, coverex_4_mg, dorithricin_mentol, dulsevia_60_mg, jutavit_c_vitamin, kalium_r, lactamed, letrox_50_mikrogramm, naprosyn_250_mg, normodipine_5_mg, ocutein, salazopyrin_en_500_mg, semicillin_500_mg, sinupret_forte, strepfen_8_75_mg, tricovel_tricoage45, urzinol, valeriana_teva, verospiron_25_mg, vitamin_d3_fresenius_1000_ne, voltaren_dolo_rapid_25_mg

4. Почему эти классы модель распознаёт без

## Итоговый вывод

Первоначальное обучение только нового классификатора при замороженной предобученной части MobileNetV3-small позволило получить accuracy 69,44%, что оказалось ниже целевого значения 75%. Поэтому был выполнен fine-tuning: последний блок признаков модели был разморожен и дополнительно обучен с уменьшенным learning rate. После fine-tuning итоговая accuracy составила 77,78%, поэтому требование проекта выполнено.